In [5]:
# Memory dump from GDB
# (gdb) x/512xb 0x00804000
memory_dump = """
0x804000:       0x39    0x92    0x9a    0xfe    0x80    0xff    0xc0    0xee
0x804008:       0xff    0xff    0xff    0xff    0xff    0xff    0xff    0xff
0x804010:       0x10    0x40    0x80    0x00    0xff    0xff    0xff    0xff
0x804018:       0xff    0xff    0xff    0xff    0xff    0xff    0xff    0xff
0x804020:       0xff    0xff    0xff    0xff    0xff    0xff    0xff    0xff
0x804028:       0xff    0xff    0xff    0xff    0xff    0xff    0xff    0xff
0x804030:       0xff    0xff    0xff    0xff    0xff    0xff    0xff    0xff
0x804038:       0xff    0xff    0xff    0xff    0xff    0xff    0xff    0xff
"""

In [ ]:
import struct

DEFAULTS = {
	'BOD33 Disable': 0x1,
	'BOD33 Level': 0x1C,
	'BOD33 Action': 0x1,
	'BOD33 Hysteresis': 0x2,
	'NVM BOOTPROT': 0xF,
	'SBLK': 0x0,
	'PSZ': 0x0,
	'RAM ECCDIS': 0x1,
	'WDT Enable': 0x0,
	'WDT Always-On': 0x0,
	'WDT Period': 0xB,
	'WDT Window': 0xB,
	'WDT EWOFFSET': 0xB,
	'WDT WEN': 0x0,
	'NVM LOCKS': 0xFFFFFFFF
}

# def extract_bits(value, offset, mask):
# 	return (value >> offset) & mask

def extract_bits(value, start, end):
	mask = (1 << (end - start + 1)) - 1
	return (value >> start) & mask


def decode_user_row(memory_dump):
	# Extract only the hexadecimal values from the memory dump
	hex_values = ''.join(line.split(':')[1].strip().replace(' ', '') for line in memory_dump.strip().split('\n'))
	hex_values = hex_values.replace('0x', '').replace('\n', '')
	
	# Convert the cleaned hex string to bytes
	memory_bytes = bytes.fromhex(hex_values)
	
	# Convert bytes to 32-bit words
	# words = struct.unpack('<8I', memory_bytes[:32])
	words = struct.unpack('<8I', memory_bytes[:32])
	
	settings = {
		'BOD33 Disable': extract_bits(words[0], 0, 0),
		'BOD33 Level': extract_bits(words[0], 1, 8),
		'BOD33 Action': extract_bits(words[0], 9, 10),
		'BOD33 Hysteresis': extract_bits(words[0], 11, 14),
		'BOD12 Calibration': extract_bits(words[0], 15, 25),
		'NVM BOOTPROT': extract_bits(words[0], 26, 29),
		'SBLK': extract_bits(words[1], 0, 3),
		'PSZ': extract_bits(words[1], 4, 6),
		'RAM ECCDIS': extract_bits(words[1], 7, 7),
		'WDT Enable': extract_bits(words[1], 16, 16),
		'WDT Always-On': extract_bits(words[1], 17, 17),
		'WDT Period': extract_bits(words[1], 18, 21),
		'WDT Window': extract_bits(words[1], 22, 25),
		'WDT EWOFFSET': extract_bits(words[1], 26, 29),
		'WDT WEN': extract_bits(words[1], 30, 30),
		'NVM LOCKS': words[2],
		'User Page': words[3]
	}
	return settings



def display_user_row_settings(settings):
	print("SAME54 User Row Settings (Current vs Default):")
	print("-" * 50)
	for key, value in settings.items():
		if key in DEFAULTS:
			if value != DEFAULTS[key]:
				print(f"{key:15}: 0x{value:X} (Default: 0x{DEFAULTS[key]:X}) *DIFFERENT*")
			else:
				print(f"{key:15}: 0x{value:X} (Default: 0x{DEFAULTS[key]:X})")
		else:
			print(f"{key:15}: 0x{value:X}")



# Decode the memory dump
decoded_settings = decode_user_row(memory_dump)

# Display the formatted settings
display_user_row_settings(decoded_settings)
